# Blue Books: collect, refresh, explore

This notebook helps you build and inspect a searchable collection from the
Blue Books material you can access. Start with **1. Open your workspace**.
Then choose the task you need; you do not have to run every section.

| Your goal | Sections to run |
|---|---|
| Inspect saved results | 1, then 2 |
| Update terms and suggested codes from saved pages | 1, then 3 |
| Collect from the website | 1, then 4; finally 3 |
| Copy the results to SQL Server | 1, then 5 |

Website steps need a browser on your own computer and your own access rights.
Saved-data work needs the local reference files and neighboring BBNotebook
helpers, but no login. Keep databases, reference material and login details
private. Use source material according to its access terms.

[First-use guide](docs/getting-started.md) |
[Everyday tasks](docs/everyday-use.md) |
[Help](docs/troubleshooting.md)

## 1. Open your workspace

Select your existing Python environment in the notebook's kernel picker.
The cell below finds the project and reads existing settings without changing
them. If the project is not found, set `PROJECT_FOLDER` to its full folder path.
If a package is missing, follow the first-use guide before continuing.

**Expected result:** the selected database path is printed. If anything fails,
stop and correct it before running a later section.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
from contextlib import closing

PROJECT_FOLDER = None  # Or r"C:\your\BBCrawler folder"
start = Path(PROJECT_FOLDER).resolve() if PROJECT_FOLDER else Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "app" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Set PROJECT_FOLDER to the BBCrawler project folder.")
if str(ROOT / "app") not in sys.path:
    sys.path.insert(0, str(ROOT / "app"))
import bluebooks_local as local
from project_paths import DEFAULT_CONFIG, load_config

CONFIG_PATH = DEFAULT_CONFIG
config = load_config(CONFIG_PATH)
DB = Path(config["db_path"])
CRAWLER_DIR = ROOT

def run_script(script, *args):
    env = os.environ.copy()
    env["PYTHONUTF8"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    command = [sys.executable, "-u", str(ROOT / "app" / script), *map(str, args)]
    with subprocess.Popen(command, cwd=ROOT, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8") as process:
        for line in process.stdout:
            print(line, end="")
        result = process.wait()
    if result:
        raise RuntimeError(f"{script} failed (exit {result}); correct the reported error.")
    return result

print("Workspace:", ROOT)
print("Database:", DB)
print("Settings loaded; no data has been changed.")

## 2. Inspect saved results

These cells read the database; they do not refresh or collect anything.
The first shows book names, short labels and editions. The second searches
saved text. Change `QUERY` to a word or phrase of interest.
The third shows terms alongside source or suggested codes.

An official reference match is a **suggestion**, not a diagnosis or an approved
code assignment. `is_proposed_code` and `is_proposed_term` distinguish suggestions
from what the source page actually says. Missing codes remain blank.
See the [data guide](docs/technical/data-guide.md) for column meanings.

In [ ]:
import pandas as pd
with closing(local.connect_sqlite(DB, readonly=True)) as connection:
    display(pd.read_sql_query(
        "SELECT book_id,title,alias,edition FROM books ORDER BY book_id", connection))

In [ ]:
QUERY = 'neuroendocrine'
with closing(local.connect_sqlite(DB, readonly=True)) as connection:
    display(pd.DataFrame(local.search(connection, QUERY, limit=10),
                         columns=["page_id", "title", "url", "full_text"]))

In [ ]:
with closing(local.connect_sqlite(DB, readonly=True)) as connection:
    display(pd.read_sql_query(
        "SELECT alias,book_name,edition,page_title,TermUS,candidateTermUS,"
        "ICDO32,ICDO4,is_proposed_code,is_proposed_term,lookup_outcome,review_status "
        "FROM v_page_code_terms ORDER BY page_id,TermUS LIMIT 30", connection))

## 3. Update from saved pages

This rebuilds terms, reference matches and search text from material already on
disk. It does not log in or fetch website pages. Required reference files must
be present. A safety copy is made before changes begin.

Set `CONFIRM_REFRESH = True` only when ready. **Expected result:** table counts
are printed. If an error occurs, stop; keep the backup and use the help guide.
After this step, SQL Server is unchanged until you run section 5.

In [ ]:
CONFIRM_REFRESH = False
if not CONFIRM_REFRESH:
    raise RuntimeError("No changes made. Set CONFIRM_REFRESH = True to update saved data.")
print(local.backup_databases(DB, ROOT / "local_backups"))
result = local.refresh_local(DB)
print(result["counts"])

## 4. Collect from the website (optional)

Skip this entire section when working with saved data. You need Playwright and
its Chromium browser installed locally; the first-use guide explains how.

The login cell opens a browser and saves your session privately. Complete login
in that browser within `login_wait_seconds` from your settings (normally two
minutes). The session is saved automatically when that time ends. You do not
need to send a terminal Enter key from the notebook.
Discovery saves the volumes you can access. Inspect the list before choosing
book IDs for collection. Collection writes pages and may download images.

Every online action requires its own confirmation. If your session expires,
log in again before resuming. Start with a small `LIMIT`; zero means all pages.
`RESCRAPE = True` replaces existing collected content for the selected books,
so leave it false unless you intend that and have a backup.

In [ ]:
CONFIRM_LOGIN = False
if not CONFIRM_LOGIN:
    raise RuntimeError("No browser opened. Set CONFIRM_LOGIN = True to log in.")
run_script("bluebooks_auth.py", "--config", CONFIG_PATH,
           "--wait-seconds", config.get("login_wait_seconds", 120))

In [ ]:
CONFIRM_DISCOVERY = False
if not CONFIRM_DISCOVERY:
    raise RuntimeError("No website access. Set CONFIRM_DISCOVERY = True to discover volumes.")
print(local.backup_databases(DB, ROOT / "local_backups") if DB.exists() else "First collection")
run_script("bluebooks_books.py", "--config", CONFIG_PATH, "discover")
run_script("bluebooks_books.py", "--config", CONFIG_PATH, "list")

In [ ]:
BOOKS = [48, 49, 50, 51]
TOC_ONLY = False
LIMIT = 0
NO_IMAGES = True
RESCRAPE = False
CONFIRM_COLLECTION = False
if not CONFIRM_COLLECTION:
    raise RuntimeError("No collection started. Set CONFIRM_COLLECTION = True when ready.")
if not BOOKS:
    raise ValueError("Choose BOOKS from the discovered inventory before collecting.")
if DB.exists():
    print(local.backup_databases(DB, ROOT / "local_backups"))
args = ["--config", CONFIG_PATH, "--book", *map(str, BOOKS)]
if TOC_ONLY:
    args.append("--toc-only")
if LIMIT:
    args.extend(["--limit", str(LIMIT)])
if NO_IMAGES:
    args.append("--no-images")
if RESCRAPE:
    args.append("--rescrape")
run_script("bluebooks_crawl.py", *args)
print("Collection finished. Run section 3 to update terms and search results.")

## 5. Copy to SQL Server (optional)

This replaces managed data in the selected SQL Server database with the current
SQLite results and checks that their values match. Activity logs are not copied.
Use an existing SQL Server database, Windows authentication and Microsoft ODBC
Driver 17 or 18. This workflow does not take SQL-login passwords from a cell.

Run section 3 first if saved pages or reference inputs changed. Check the server
and database carefully, then set `CONFIRM_MIRROR = True`. Both databases are
backed up before replacement. **Expected result:** all table checks say `True`.
On failure, stop and keep the reported backups; do not repeatedly replace data.

In [ ]:
SQLSERVER = '(local)'
DATABASE = 'BlueBooks'
CONFIRM_MIRROR = False
if not CONFIRM_MIRROR:
    raise RuntimeError("SQL Server unchanged. Set CONFIRM_MIRROR = True when ready.")
print(local.backup_databases(DB, ROOT / "local_backups", SQLSERVER, DATABASE))
checks = local.mirror(DB, SQLSERVER, DATABASE, include_book70=False)
print({table: check["match"] for table, check in checks.items()})
with closing(local.connect_server(SQLSERVER, DATABASE)) as connection:
    print(local.server_search(connection))

## Where to go next

- [Everyday use](docs/everyday-use.md): choose a repeatable workflow.
- [Troubleshooting](docs/troubleshooting.md): resolve setup and connection issues.
- [Data guide](docs/technical/data-guide.md): understand source text, suggestions
  and saved review decisions.
- The neighboring BBNotebook project offers more detailed exploration notebooks.

Additional reader cells are preserved below when this notebook is regenerated.
They are not part of the standard workflow; inspect them before running them.